# Voice Interaction

by Tobias Erbacher

This notebook has been developed to run locally. If you want to use Colab this is not guaranteed to work.

Let's import the libraries necessary to run this notebook:

In [1]:
import math
import scipy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import time
import torch

# Speech-to-Text
import librosa
import sounddevice as sd
from scipy.io.wavfile import write
import queue
from pynput import keyboard
import whisper

# Text-to-Speech
import pyttsx3 # Model 1

import unidecode # Model 2
import inflect 

---

### Speech-to-Text

In [2]:
MAX_DURATION = 30                               # Maximum audio recording duration in seconds.
SAVE_AUDIO = False                              # True: Audio will be saved after recording; False: Audio will not be saved after recording.
OUTPUT_FILENAME = "speech_input.wav"            # Name of the audio file that will be saved. Remember to give it a filetype, i.e. ".wav" at the end.
PLAY_AFTER_RECORDING = False                    # Do you wish to replay the audio after the recording?
AUDIO_RECORDING_STOP_KEY = keyboard.Key.space   # The key that stops the audio recording. Default is space bar, i.e. keyboard.Key.space.

In [3]:
SAMPLING_RATE = 16000                           # Hz. Note that openai whisper requires 160000 Hz
NUMBER_OF_RECORDING_CHANNELS = 1                # Number of channels (1 for mono, 2 for stereo). Note that openai whisper requires mono channel.

WAVEFORM_PARAMS = {                             # Plot parameters for the wave form.
    'figure.figsize': (16, 6),
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'grid.alpha': 0.5,
    'legend.fontsize': 12,
    #'legend.frameon': False,
    'axes.grid': True,
    'axes.labelsize': 12,
    'lines.linewidth': 0.5,
    # Font settings for LaTeX
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern'],
}

MEL_SPECTROGRAM_WINDOW_SIZE = 0.025             # Mel spectrogram window size in seconds.
MEL_SPECTROGRAM_HOP_SIZE = 0.01                 # Mel spectrogram hop size (delay between consecutive windows) in seconds.
MEL_SPECTROGRAM_START_TIME = 0                  # Mel spectrogram start time in seconds.
N_FFT = 1300                                    # Mel spectrogram sample number for Fast Fourier Transform.
N_MEL = 80                                      # Mel spectrogram band number to generate.

MEL_SPECTROGRAM_PARAMS = {                      # Plot parameters for the wave form.
    'figure.figsize': (16, 6),
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'grid.alpha': 0.5,
    'legend.fontsize': 12,
    #'legend.frameon': False,
    'axes.grid': True,
    'axes.labelsize': 12,
    'lines.linewidth': 0.5,
    # Font settings for LaTeX
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern'],
}

First we need to set up the audio recorder. We do this with an InputStream and a buffer.

In [ ]:
audio_recording_queue = queue.Queue()

def audio_callback(input_data, frames, time, status): # Do not remove the not needed parameters as they are somehow used in the InputStream
    if status:
        print(status)
    audio_recording_queue.put(input_data.copy())

def on_press(key):
    global stop_recording
    if key == AUDIO_RECORDING_STOP_KEY:
        stop_recording = True
        key_listener.stop()
        print("Recording has been stopped by the user.")

AUDIO_RECORDING_START_TIME = time.time()
audio_recording_buffer = []
stop_recording = False

print("Speak now...")
with sd.InputStream(samplerate=SAMPLING_RATE, channels=NUMBER_OF_RECORDING_CHANNELS, callback=audio_callback):
    key_listener = keyboard.Listener(on_press=on_press)
    key_listener.start()

    while not stop_recording:
        if time.time() - AUDIO_RECORDING_START_TIME < MAX_DURATION:
            audio_recording_buffer.append(audio_recording_queue.get())
        else:
            print("Recording has been stopped due to maximum length.")
            key_listener.stop()
            break

AUDIO_RECORDING_FORMATTED = np.concatenate(audio_recording_buffer, axis=0).squeeze()

VOICE_INPUT = AUDIO_RECORDING_FORMATTED / np.max(np.abs(AUDIO_RECORDING_FORMATTED))
      
if SAVE_AUDIO:
    write(OUTPUT_FILENAME, SAMPLING_RATE, (VOICE_INPUT * 32767).astype(np.int16))

if PLAY_AFTER_RECORDING:
    sd.play(VOICE_INPUT, samplerate=SAMPLING_RATE)
    sd.wait()

Now that we have recorded the audio, we can put it into a speech-to-text model of our choice. We start with the openai-whisper models:

In [5]:
model = whisper.load_model("base")#.to("cuda") # Move the model to the GPU if available.         

Next, we can detect what language was spoken:

In [ ]:
_, LANGUAGE_PROBABILITIES = model.detect_language(whisper.log_mel_spectrogram(whisper.pad_or_trim(VOICE_INPUT)).to(model.device))
DETECTED_LANGUAGE = max(LANGUAGE_PROBABILITIES, key=LANGUAGE_PROBABILITIES.get)
print("Detected language: " + str(DETECTED_LANGUAGE) + ", confidence: " + str(round(LANGUAGE_PROBABILITIES[DETECTED_LANGUAGE] * 100, 2)))

Thereupon, we transform the audio to text.

In [ ]:
TEXT = model.transcribe(VOICE_INPUT)["text"]

In [ ]:
TEXT

### Speech-to-Text Analytics

If you wish to perform some analytics on the recorded voice input, you can run the following cells. First, you can take a look at the wave form.

In [ ]:
DURATION = VOICE_INPUT.shape[0] / SAMPLING_RATE
TIME = np.linspace(0, DURATION, VOICE_INPUT.shape[0])

mpl.rcParams.update(WAVEFORM_PARAMS)
plt.plot(TIME, VOICE_INPUT)
plt.title("Recorded Audio Waveform")
plt.xlabel("time [s]")
plt.ylabel("Amplitude [a.u.]")
plt.show()

Next up, we can plot the Mel-spectrogram:

In [ ]:
W = int(math.ceil(MEL_SPECTROGRAM_WINDOW_SIZE * SAMPLING_RATE))
D = int(math.ceil(MEL_SPECTROGRAM_HOP_SIZE * SAMPLING_RATE))
T = int(math.ceil(MEL_SPECTROGRAM_START_TIME * SAMPLING_RATE))

MEL_SPECTROGRAM = librosa.feature.melspectrogram(y=VOICE_INPUT, sr=SAMPLING_RATE, n_fft=N_FFT, win_length=W, hop_length=D, n_mels=N_MEL).squeeze()

MEL_SPECTROGRAM_FREQUENCY_BAND_BIN_EDGES = np.arange(MEL_SPECTROGRAM.shape[0] + 1)
MEL_SPECTROGRAM_TIME_SLICES = np.linspace(0, DURATION, MEL_SPECTROGRAM.shape[1] + 1)
MEL_SPECTROGRAM_POWER = librosa.power_to_db(MEL_SPECTROGRAM, ref=np.max)

MEL_FREQUENCIES = np.append(librosa.mel_frequencies(n_mels=N_MEL, fmax=SAMPLING_RATE / 2), SAMPLING_RATE / 2)

mpl.rcParams.update(MEL_SPECTROGRAM_PARAMS)
#fig = plt.figure(figsize=(10,4))
plt.pcolormesh(MEL_SPECTROGRAM_TIME_SLICES, MEL_SPECTROGRAM_FREQUENCY_BAND_BIN_EDGES, MEL_SPECTROGRAM_POWER)
plt.colorbar(label='Power [dB]')
plt.title("Mel Spectrogram of Recorded Audio")
plt.xlabel('Time [s]')
plt.ylabel('Frequency [Hz]')
plt.yticks(ticks=MEL_SPECTROGRAM_FREQUENCY_BAND_BIN_EDGES[9::10], labels=[f'{freq:.0f}' for freq in MEL_FREQUENCIES[9::10]])
plt.show()

---

### Text-to-Speech

##### Model 1

In [ ]:
PYTTSX3_ENGINE = pyttsx3.init()
PYTTSX3_VOICES = PYTTSX3_ENGINE.getProperty('voices')

First, choose the voice that you want to hear. Default is 0. To show the available voices, run the following cell:

In [ ]:
for idx, voice in enumerate(PYTTSX3_VOICES):
        print("Voice: " + str(idx) + ", ID: " + str(voice.id) + ", Name: " + str(voice.name) + ", Language: " + str(voice.languages))

Now, modify $\texttt{PYTTSX3\_VOICE}$ to be the voice ID that you want.

In [ ]:
PYTTSX3_VOICE = 0
PYTTSX3_TALKING_SPEED = 150
PYTTSX3_VOLUME = 1

Next, run the cell below to hear the system's response to your question.

In [5]:
PYTTSX3_ENGINE.setProperty('rate', PYTTSX3_TALKING_SPEED)
PYTTSX3_ENGINE.setProperty('volume', PYTTSX3_VOLUME)

PYTTSX3_ENGINE.setProperty('voice', PYTTSX3_VOICES[PYTTSX3_VOICE].id)
PYTTSX3_ENGINE.say(TEXT)
PYTTSX3_ENGINE.runAndWait()

To hear the samples, execute the following cell:

In [ ]:
def pyttsx3_samples():
    # Some rather interesting voice samples...
    BAD_NEWS = "da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da"
    IN_THE_HALL_OF_THE_MOUNTAIN_KING = "da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da"
    BAAAH = "aaaa aaaa aaaa aaaa"
    BOING = "boing boing boing boing boing boing boing"
    BLUB = "blub blub blub blub blub blub blub blub blub"
    DERANGED = "ha ha ha ha ha ha ha ha ha"
    LAND_OF_HOPE_AND_GLORY =  "da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da"
    HAHA = "ha ha ha ha ha ha ha ha ha ha ha ha ha"
    ORGAN =  "da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da da"
    TRUCK = "weeeeeed weeeeeed weeeeeed"
    SKIBIDI = "I'm the Scatman, skibydibby dibyo dadubdubyo, I'm the Scatman, skibidibby dibyo dadubdubyo dadubdub"

    SAMPLES = [
        (BAD_NEWS, 6), 
        (IN_THE_HALL_OF_THE_MOUNTAIN_KING, 12), 
        (BAAAH, 7), 
        (BOING, 9), 
        (BLUB, 10), 
        (DERANGED, 16), 
        (LAND_OF_HOPE_AND_GLORY, 47), 
        (HAHA, 76), 
        (ORGAN, 100), 
        (TRUCK, 168), 
        (SKIBIDI, 173)
    ]

    for item in SAMPLES:
        PYTTSX3_ENGINE.setProperty('voice', PYTTSX3_VOICES[item[1]].id)
        PYTTSX3_ENGINE.say(item[0])
        PYTTSX3_ENGINE.runAndWait()

pyttsx3_samples()

##### Model 2

In [2]:
PYTTSX3_TALKING_SPEED = 22050

In [ ]:
TACOTRON2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', model_math='fp32', map_location='cpu')#.to('cuda')  # Move the model to the GPU if available.

WAVEGLOW = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', model_math='fp32', map_location='cpu')
WAVEGLOW = WAVEGLOW.remove_weightnorm(WAVEGLOW)#.to('cuda')  # Move the model to the GPU if available.

TEXT = "And you know what they call a Quarter Pounder with Cheese in Paris?"

TTS_UTILS = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tts_utils')
WAVEGLOW_SEQUENCES, WAVEGLOW_LENGTHS = TTS_UTILS.prepare_input_sequence([TEXT])

with torch.no_grad():
    TACOTRON2_MEL, _, _ = TACOTRON2.infer(WAVEGLOW_SEQUENCES, WAVEGLOW_LENGTHS)
    TACOTRON2_VOICE = WAVEGLOW.infer(TACOTRON2_MEL)[0].data.cpu().numpy()

sd.play(TACOTRON2_VOICE, samplerate=PYTTSX3_TALKING_SPEED)
sd.wait()